In [1]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
import glob, os

# find every season file
files = sorted(glob.glob("data/E2_*.csv"))
print("Found files:", files)

season_frames = []
for f in files:
    # extract the season label from the filename, e.g. "E2_24-25.csv" -> "24-25"
    season = os.path.basename(f).replace("E2_", "").replace(".csv", "")
    s = pd.read_csv(f)
    s = s[["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG"]].copy()
    s["Season"] = season
    s["Date"] = pd.to_datetime(s["Date"], dayfirst=True, errors="coerce")
    season_frames.append(s)
    print(f"{season}: {len(s)} matches, {s['Date'].min().date()} to {s['Date'].max().date()}")

# stack them into one dataset
allmatches = pd.concat(season_frames, ignore_index=True)
print(f"\nTotal: {len(allmatches)} matches across {len(files)} seasons")

Found files: ['data/E2_20-21.csv', 'data/E2_21-22.csv', 'data/E2_22-23.csv', 'data/E2_23-24.csv', 'data/E2_24-25.csv', 'data/E2_25-26.csv']
20-21: 552 matches, 2020-09-12 to 2021-05-09
21-22: 552 matches, 2021-08-07 to 2022-04-30
22-23: 552 matches, 2022-07-30 to 2023-05-07
23-24: 552 matches, 2023-08-05 to 2024-04-27
24-25: 552 matches, 2024-08-09 to 2025-05-03
25-26: 552 matches, 2025-08-01 to 2026-05-02

Total: 3312 matches across 6 seasons


In [3]:
# how many distinct teams across all six seasons?
all_teams = pd.unique(allmatches[["HomeTeam", "AwayTeam"]].values.ravel())
print(f"Distinct teams across 6 seasons: {len(all_teams)}")

# how many seasons does each team appear in?
team_seasons = (allmatches.groupby("HomeTeam")["Season"]
                .nunique()
                .sort_values(ascending=False))
print("\nSeasons each team appears in:")
print(team_seasons)

Distinct teams across 6 seasons: 49

Seasons each team appears in:
HomeTeam
Lincoln               6
Burton                6
Bolton                5
Peterboro             5
Shrewsbury            5
Wycombe               5
Charlton              5
Wigan                 5
Fleetwood Town        4
Blackpool             4
Cambridge             4
Northampton           4
Oxford                4
Portsmouth            4
Bristol Rvs           4
Plymouth              4
Barnsley              4
Exeter                4
Milton Keynes Dons    3
Stevenage             3
Reading               3
Leyton Orient         3
Rotherham             3
Port Vale             3
AFC Wimbledon         3
Accrington            3
Doncaster             3
Cheltenham            3
Ipswich               3
Crewe                 2
Sunderland            2
Stockport             2
Sheffield Weds        2
Huddersfield          2
Mansfield             2
Derby                 2
Gillingham            2
Morecambe             2
Crawley Town

In [5]:
def backtest_one_season(season_df):
    # sort by date and split 70/30 within THIS season only
    s = season_df.sort_values("Date").reset_index(drop=True)
    split = int(len(s) * 0.70)
    tr, te = s.iloc[:split], s.iloc[split:]

    # strengths from this season's training portion only
    ahg, aag = tr["FTHG"].mean(), tr["FTAG"].mean()
    h = tr.groupby("HomeTeam").agg(hs=("FTHG","mean"), hc=("FTAG","mean"))
    a = tr.groupby("AwayTeam").agg(as_=("FTAG","mean"), ac=("FTHG","mean"))
    st = h.join(a)

    def predict(ht, at):
        he = st.loc[ht,"hs"] * st.loc[at,"ac"] / ahg
        ae = st.loc[at,"as_"] * st.loc[ht,"hc"] / aag
        hw, dr, aw = 0.0, 0.0, 0.0
        for hg in range(10):
            for ag in range(10):
                p = poisson.pmf(hg, he) * poisson.pmf(ag, ae)
                if hg > ag: hw += p
                elif hg == ag: dr += p
                else: aw += p
        return hw, dr, aw

    out = []
    for _, r in te.iterrows():
        ht, at = r["HomeTeam"], r["AwayTeam"]
        if ht not in st.index or at not in st.index:
            continue
        hw, dr, aw = predict(ht, at)
        actual = "H" if r["FTHG"]>r["FTAG"] else ("A" if r["FTHG"]<r["FTAG"] else "D")
        out.append({"p_home":hw, "p_draw":dr, "p_away":aw, "actual":actual})
    return pd.DataFrame(out)

# run it for every season and pool
pieces = []
for season, sdf in allmatches.groupby("Season"):
    res = backtest_one_season(sdf)
    res["Season"] = season
    pieces.append(res)
    print(f"{season}: {len(res)} test predictions")

results = pd.concat(pieces, ignore_index=True)
print(f"\nPooled test predictions: {len(results)}")

20-21: 166 test predictions
21-22: 166 test predictions
22-23: 166 test predictions
23-24: 166 test predictions
24-25: 166 test predictions
25-26: 166 test predictions

Pooled test predictions: 996


In [7]:
results["home_won"] = (results["actual"] == "H").astype(int)

bins = [0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 1.0]
results["bucket"] = pd.cut(results["p_home"], bins)

calib = results.groupby("bucket", observed=True).agg(
    n=("home_won", "size"),
    avg_predicted=("p_home", "mean"),
    actual_rate=("home_won", "mean")
)
calib

,n,avg_predicted,actual_rate
bucket,,,
"(0.0, 0.2]",113,0.145506,0.238938
"(0.2, 0.3]",159,0.250497,0.226415
"(0.3, 0.4]",181,0.349483,0.359116
"(0.4, 0.5]",191,0.450878,0.445026
"(0.5, 0.6]",159,0.545251,0.515723
"(0.6, 0.7]",127,0.645516,0.551181
"(0.7, 1.0]",66,0.762082,0.712121
